# Accounts - Bronze

**Reads:** `bronze.bronze_accounts`  
**Writes:** `validate.bronze_clean_accounts` (overwrite)

## Stakeholders Requests
- Trim and Upper Case business key attribute.
- Standardize category attribute.
- Standardize Date Format in all date attributes.
- Parse Interest Rates
- Mask Credit Card Number and Implement Luhn Checksum for validation
- Finally remove multiple customers updates and keep only latest customer updates using 'account_id' & 'updated_at' as key

## Imports

In [ ]:
# Imports
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructField, StructType

## Shared functions

In [ ]:
%run NB_FUNCTIONS

## Loading Dataframe in Spark

In [ ]:
acc_df = spark.read.table("bronze.bronze_accounts")

In [ ]:
display(acc_df)

## Removing Null Spaces or Tokens from DF

In [ ]:
acc_n = normalise_null(acc_df)

## Standardizing Category, Trimming and Casing

In [ ]:
ACC_TYPE = {
    "SAVINGS": "SAVINGS",
    "SB": "SAVINGS",
    "CURRENT": "CURRENT",
    "CA": "CURRENT",
    "CREDIT_CARD": "CREDIT_CARD",
    "CREDIT CARD": "CREDIT_CARD",
    "CC": "CREDIT_CARD",
    "FIXED_DEPOSIT": "FIXED_DEPOSIT",
    "FD": "FIXED_DEPOSIT",
    "FIXED DEPOSIT": "FIXED_DEPOSIT",
}
COUNTRY = {"IN": "IN", "INDIA": "IN", "IND": "IN"}

CURRENCY = {
    "INR": "INR",
    "RS": "INR",
    "₹": "INR",
    "USD": "USD",
    "$": "USD",
    "US$": "USD",
    "EUR": "EUR",
    "GBP": "GBP",
    "AED": "AED",
    "SGD": "SGD",
}

acc_c = (
    acc_n.withColumn("customer_id", F.upper(F.trim("customer_id")))
    .withColumn("account_type", mapping_values("account_type", ACC_TYPE))
    .withColumn("ifsc_code", F.upper(F.trim("ifsc_code")))
    .withColumn("currency", mapping_values("currency", CURRENCY))
    .withColumn("open_date", date_parse("open_date"))
    .withColumn("closed_date", date_parse("closed_date"))
    .withColumn("status", F.upper(F.trim("status")))
    .withColumn("credit_limit", amount_parse("credit_limit"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

## Masking and Luhn Checksum Credit Card Number

In [ ]:
card = F.regexp_replace("card_number", r"[\s-]", "")
reverse = F.reverse(
    F.split(card, "")
)  # Im doing this because luhn algo reads from right to left
doubled = F.transform(
    reverse,
    lambda d, i: F.when(
        i % 2 == 1,
        F.when(d.cast("int") * 2 > 9, d.cast("int") * 2 - 9).otherwise(
            d.cast("int") * 2
        ),
    ).otherwise(d.cast("int")),
)

# Luhn Checksum, returns True/False
luhn_ok = F.aggregate(doubled, F.lit(0), lambda accum, x: accum + x) % 10 == 0

acc_c = (
    acc_c.withColumn(
        "card_valid",
        F.when(F.col("card_number").isNull(), None).otherwise(
            card.rlike(r"^\d{13,19}$") & luhn_ok
        ),
    )
    .withColumn(
        "card_last4",
        F.when(F.col("card_number").isNotNull(), F.substring("card_number", -4, 4)),
    )
    .drop("card_number")
)

## Parsing Interest Rate

In [ ]:
acc_c = acc_c.withColumn("interest_rate", interest_rate_parse("interest_rate"))

## Keeping only latest updated record only

In [ ]:
acc_l = latest_by_key(acc_c.dropDuplicates(), ["account_id"], ["updated_at"])

## Processing to Silver Layer

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS validate")
acc_l.write.mode("overwrite").format("delta").saveAsTable(
    "validate.bronze_clean_accounts"
)